# Benchmark de Métodos de Busca com Vespa


Este notebook executa testes comparando diferentes métodos de recuperação de documentos usando o Vespa:
- **BM25**
- **Busca Semântica**
- **Busca Híbrida**
- **Fusão Linear**
Inclui medições de tempo usando `timing` do Vespa e métricas de avaliação.


In [39]:

import sys, os, json, numpy as np, pandas as pd
from sentence_transformers import SentenceTransformer
from vespa.application import Vespa
from tqdm import tqdm

from src.metrics import mrr_score, map_score, mr_score, mf1_score, mndcg_score

# Caminho dos módulos locais
src_path = os.path.abspath(os.path.join(os.getcwd(), "src"))
if src_path not in sys.path:
    sys.path.insert(0, src_path)

from retrivers.vespa_retrievers import VespaBM25

model = SentenceTransformer("intfloat/e5-small-v2")
vespa = Vespa(url="http://localhost", port=8080)
vespa_retriever = VespaBM25()


In [ ]:
import pandas as pd
import json

# Carregue seu DataFrame
df = pd.read_pickle("/home/lunardonbruno/msmarco/subset_msmarco_train_0/subset_msmarco_train_0.01_9.pkl")

documents_df = pd.DataFrame([
    {"doc_id": d.doc_id, "content": d.text}
    for d in df["docs"].values()
])

documents_df=documents_df.sample(frac=0.01, random_state=1, replace=False)


# Caminho de saída
jsonl_path = "tutorials/vespa/app-vespa/msmarco_feed.jsonl"

with open(jsonl_path, "w") as f:
    for row in documents_df.itertuples():
        feed_item = {
            "put": f"id:msmarco:msmarco::{row.doc_id}",
            "fields": {
                "id": row.doc_id,
                "content": row.content
            }
        }
        f.write(json.dumps(feed_item) + "\n")

print(f"✅ Arquivo JSONL criado com sucesso em: {jsonl_path}")

✅ Arquivo JSONL criado com sucesso em: tutorials/vespa/app-vespa/msmarco_feed.jsonl


In [41]:
import subprocess

command = [
    "docker", "exec", "vespa", "vespa", "feed",
    "/app-vespa/msmarco_feed.jsonl"
]

process = subprocess.run(command, capture_output=True, text=True)

print("✅ Feed concluído.")
print("STDOUT:")
print(process.stdout)
print("STDERR:")
print(process.stderr)


✅ Feed concluído.
STDOUT:
{
  "feeder.operation.count": 278,
  "feeder.seconds": 19.439,
  "feeder.ok.count": 278,
  "feeder.ok.rate": 14.301,
  "feeder.error.count": 0,
  "feeder.inflight.count": 0,
  "http.request.count": 278,
  "http.request.bytes": 96195,
  "http.request.MBps": 0.005,
  "http.exception.count": 0,
  "http.response.count": 278,
  "http.response.bytes": 36610,
  "http.response.MBps": 0.002,
  "http.response.error.count": 0,
  "http.response.latency.millis.min": 721,
  "http.response.latency.millis.avg": 1361,
  "http.response.latency.millis.max": 2321,
  "http.response.code.counts": {
    "200": 278
  }
}

STDERR:



In [44]:

from vespa.application import Vespa

class VespaRetriever:
    def __init__(self, ranking="bm25"):
        self.app = Vespa(url="http://localhost", port=8080)
        self.ranking = ranking

    def run(self, query_text, k=10):
        # Option 1: Use userQuery() with proper query parameter
        query_body = {
            "yql": "select * from sources * where userQuery();",
            "query": query_text,  # This is the key - userQuery() uses this parameter
            "ranking": self.ranking,
            "hits": k,
            "type": "any"  # This helps with query processing
        }
        
        # Option 2: Alternative approach using weakAnd for better recall
        # query_body = {
        #     "yql": f"select * from sources * where weakAnd(content contains '{query_text}');",
        #     "ranking": self.ranking,
        #     "hits": k
        # }

        response = self.app.query(body=query_body)
        res_json = response.get_json()

        # Check for errors
        if 'errors' in res_json:
            print(f"Query error: {res_json['errors']}")
            return [], -1.0

        try:
            vespa_time = res_json["root"]["timing"]["total"]
        except KeyError:
            print(f"⚠️  [VespaRetriever.run] Warning: 'timing' not found for ranking {self.ranking}.")
            vespa_time = -1.0

        hits = [(hit["fields"]["id"], hit.get("relevance", 1.0)) for hit in res_json.get("root", {}).get("children", [])]
        return hits, vespa_time

# Alternative version with more robust query handling
class VespaRetrieverRobust:
    def __init__(self, ranking="bm25"):
        self.app = Vespa(url="http://localhost", port=8080)
        self.ranking = ranking

    def run(self, query_text, k=10):
        # Try different query approaches based on ranking
        if self.ranking == "bm25":
            # For BM25, use weakAnd for better recall
            query_body = {
                "yql": f"select * from sources * where weakAnd(content contains '{query_text}');",
                "ranking": self.ranking,
                "hits": k
            }
        else:
            # For semantic/hybrid, use userQuery
            query_body = {
                "yql": "select * from sources * where userQuery();",
                "query": query_text,
                "ranking": self.ranking,
                "hits": k,
                "type": "any"
            }

        response = self.app.query(body=query_body)
        res_json = response.get_json()

        # Debug: print first query result
        if hasattr(self, '_debug_first') and not self._debug_first:
            print(f"First query result for '{query_text}': {len(res_json.get('root', {}).get('children', []))} hits")
            self._debug_first = True

        try:
            vespa_time = res_json["root"]["timing"]["total"]
        except KeyError:
            vespa_time = -1.0

        hits = [(hit["fields"]["id"], hit.get("relevance", 1.0)) for hit in res_json.get("root", {}).get("children", [])]
        return hits, vespa_time

# Test the fixed version
def test_fixed_retriever():
    vespa_bm25_fixed = VespaRetriever(ranking="bm25")
    
    # Test with a sample query
    test_query = "what is machine learning"
    hits, time_taken = vespa_bm25_fixed.run(test_query, k=5)
    
    print(f"Query: '{test_query}'")
    print(f"Hits: {len(hits)}")
    print(f"Time: {time_taken}")
    if hits:
        print(f"First hit: {hits[0]}")
    
    return hits

In [57]:
# Your current retriever
vespa_bm25 = VespaRetriever(ranking="bm25")

print("=== Testing Current Retriever Behavior ===")

# Test 1: Multi-term query
query1 = "machine learning"
hits1, _ = vespa_bm25.run(query1, k=5)
print(f"Query '{query1}': {len(hits1)} hits")

# Test 2: Individual terms from the same query
hits2, _ = vespa_bm25.run("machine", k=5)
hits3, _ = vespa_bm25.run("learning", k=5)
print(f"Query 'machine': {len(hits2)} hits")
print(f"Query 'learning': {len(hits3)} hits")

# Test 3: Explicit OR comparison
query_body_or = {
    "yql": "select * from sources * where content contains 'machine' OR content contains 'learning';",
    "ranking": "bm25",
    "hits": 5
}
response_or = vespa_bm25.app.query(body=query_body_or)
hits_or = len(response_or.get_json().get('root', {}).get('children', []))
print(f"Explicit OR 'machine OR learning': {hits_or} hits")

# Test 4: Check if it's doing phrase search
phrase_query = {
    "yql": "select * from sources * where content contains phrase('machine', 'learning');",
    "ranking": "bm25", 
    "hits": 5
}
response_phrase = vespa_bm25.app.query(body=phrase_query)
hits_phrase = len(response_phrase.get_json().get('root', {}).get('children', []))
print(f"Explicit phrase search: {hits_phrase} hits")

print("\n=== Analysis ===")
if len(hits1) == 0 and (len(hits2) > 0 or len(hits3) > 0):
    print("❌ Your retriever is likely doing phrase search instead of term-based BM25")
elif len(hits1) > 0 and len(hits1) <= hits_or:
    print("✅ Your retriever appears to be working correctly (OR behavior)")
elif len(hits1) == hits_phrase:
    print("❌ Your retriever is doing exact phrase matching")
else:
    print("🤔 Mixed behavior - need more investigation")

# Test 5: Try a query from your actual dataset
print(f"\n=== Test with actual query ===")
if 'queries_df' in globals():
    sample_query = queries_df.iloc[0]['text']
    hits_sample, _ = vespa_bm25.run(sample_query, k=5)
    print(f"Sample query: '{sample_query[:50]}...'")
    print(f"Hits: {len(hits_sample)}")

=== Testing Current Retriever Behavior ===
⚠️  [VespaRetriever.run] Warning: 'timing' not found for ranking bm25.
Query 'machine learning': 0 hits
⚠️  [VespaRetriever.run] Warning: 'timing' not found for ranking bm25.
⚠️  [VespaRetriever.run] Warning: 'timing' not found for ranking bm25.
Query 'machine': 0 hits
Query 'learning': 0 hits
Explicit OR 'machine OR learning': 3 hits
Explicit phrase search: 0 hits

=== Analysis ===
❌ Your retriever is doing exact phrase matching

=== Test with actual query ===
⚠️  [VespaRetriever.run] Warning: 'timing' not found for ranking bm25.
Sample query: 'how long till i can bath after a tattoo...'
Hits: 0


In [63]:
from vespa.application import Vespa
import re

class VespaRetrieverlunas:
    def __init__(self, ranking="bm25"):
        self.app = Vespa(url="http://localhost", port=8080)
        self.ranking = ranking

    def run(self, query_text, k=10):
        if self.ranking == "bm25":
            # For BM25: use explicit OR query (we know this works!)
            # Clean and split the query into terms
            query_terms = self._clean_and_split_query(query_text)
            
            if not query_terms:
                return [], -1.0
            
            if len(query_terms) == 1:
                # Single term
                yql_query = f"select * from sources * where content contains '{query_terms[0]}';"
            else:
                # Multiple terms with OR logic (proper BM25 behavior)
                term_conditions = [f"content contains '{term}'" for term in query_terms]
                yql_query = f"select * from sources * where {' OR '.join(term_conditions)};"
            
            query_body = {
                "yql": yql_query,
                "ranking": self.ranking,
                "hits": k
            }
            
        else:
            # For semantic/hybrid rankings, you'd need embeddings
            # This is a placeholder - won't work without proper embeddings
            query_body = {
                "yql": "select * from sources * where userQuery();",
                "query": query_text,
                "ranking": self.ranking,
                "hits": k,
                "type": "any"
            }

        response = self.app.query(body=query_body)
        res_json = response.get_json()

        # Check for errors
        if 'errors' in res_json:
            print(f"Query error: {res_json['errors']}")
            return [], -1.0

        try:
            vespa_time = res_json["root"]["timing"]["total"]
        except KeyError:
            vespa_time = -1.0

        hits = [(hit["fields"]["id"], hit.get("relevance", 1.0)) for hit in res_json.get("root", {}).get("children", [])]
        return hits, vespa_time

    def _clean_and_split_query(self, query_text):
        """Clean query and split into terms for BM25"""
        # Remove extra whitespace and convert to lowercase
        query_text = query_text.strip().lower()
        
        # Remove punctuation and split into words
        # Keep only alphanumeric characters and spaces
        cleaned = re.sub(r'[^a-zA-Z0-9\s]', ' ', query_text)
        
        # Split into terms and remove empty strings
        terms = [term.strip() for term in cleaned.split() if term.strip()]
        
        # Remove very short terms (optional - you might want to keep them)
        terms = [term for term in terms if len(term) >= 2]
        
        return terms

In [64]:
def test_fixed_retriever():
    print("=== Testing Fixed Retriever ===")
    
    vespa_fixed = VespaRetrieverlunas(ranking="bm25")
    
    test_queries = [
        "machine learning",
        "how long till i can bath after a tattoo",
        "artificial intelligence",
        "the",
        "computer science"
    ]
    
    for query in test_queries:
        hits, time_taken = vespa_fixed.run(query, k=5)
        print(f"Query: '{query}' -> {len(hits)} hits")
        if hits:
            print(f"  Top relevance: {hits[0][1]:.4f}")


In [65]:
test_fixed_retriever()

=== Testing Fixed Retriever ===
Query: 'machine learning' -> 3 hits
  Top relevance: 4.7518
Query: 'how long till i can bath after a tattoo' -> 5 hits
  Top relevance: 8.3811
Query: 'artificial intelligence' -> 3 hits
  Top relevance: 7.1550
Query: 'the' -> 5 hits
  Top relevance: 0.2997
Query: 'computer science' -> 5 hits
  Top relevance: 5.0703


In [66]:
queries_df = pd.DataFrame([{"query_id": q.query_id, "text": q.text} for q in df["queries"].values()])
qrels_df = pd.DataFrame([{"query_id": q.query_id, "doc_id": q.doc_id} for q in df["qrels"]])
relevant_dict = qrels_df.groupby("query_id")["doc_id"].apply(list).to_dict()


In [ ]:
from vespa.application import Vespa

class VespaRetriever:
    def __init__(self, ranking="bm25"):
        self.app = Vespa(url="http://localhost", port=8080)
        self.ranking = ranking

    def run(self, query_text, k=10):
        query_body = {
            "yql": "select * from sources * where userQuery();",
            "query": query_text,
            "ranking": self.ranking,
            "hits": k
        }

        response = self.app.query(body=query_body)
        res_json = response.get_json()

        try:
            vespa_time = res_json["root"]["timing"]["total"]
        except KeyError:
            print(f"⚠️  [VespaRetriever.run] Warning: 'timing' not found for ranking {self.ranking}.")
            vespa_time = -1.0

        hits = [(hit["fields"]["id"], hit.get("relevance", 1.0)) for hit in res_json.get("root", {}).get("children", [])]
        return hits, vespa_time




class VespaSemanticRetriever:
    def __init__(self):
        self.app = Vespa(url="http://localhost", port=8080)

    def run(self, query_text, k=10):
        query_body = {
            "yql": "select * from sources * where ([{\"targetNumHits\":%d}]nearestNeighbor(content_embedding, query_embedding));" % k,
            "input.query(query_embedding)": query_text,
            "ranking": "semantic",
            "hits": k
        }

        response = self.app.query(body=query_body)
        res_json = response.get_json()

        try:
            vespa_time = res_json["root"]["timing"]["total"]
        except KeyError:
            print("⚠️  [VespaSemanticRetriever.run] Warning: 'timing' not found.")
            vespa_time = -1.0

        hits = [(hit["fields"]["id"], hit.get("relevance", 1.0)) for hit in res_json.get("root", {}).get("children", [])]
        return hits, vespa_time



class VespaHybridRetriever:
    def __init__(self):
        self.app = Vespa(url="http://localhost", port=8080)

    def run(self, query_text, k=10):
        query_body = {
            "yql": """
                select * from sources * where 
                ([{{"targetNumHits":{0}}}]nearestNeighbor(content_embedding, query_embedding)) 
                or userQuery();
            """.format(k),
            "input.query(query_embedding)": query_text,
            "query": query_text,
            "ranking": "hybrid",
            "hits": k
        }

        response = self.app.query(body=query_body)
        res_json = response.get_json()

        try:
            vespa_time = res_json["root"]["timing"]["total"]
        except KeyError:
            print("⚠️  [VespaHybridRetriever.run] Warning: 'timing' not found.")
            vespa_time = -1.0

        hits = [(hit["fields"]["id"], hit.get("relevance", 1.0)) for hit in res_json.get("root", {}).get("children", [])]
        return hits, vespa_time



class VespaLinearFusionRetriever:
    def __init__(self, bm25_weight=0.5, semantic_weight=0.5):
        self.app = Vespa(url="http://localhost", port=8080)
        self.bm25_weight = bm25_weight
        self.semantic_weight = semantic_weight

    def run(self, query_text, k=10):
        query_body = {
            "yql": """
                select * from sources * where 
                ([{{"targetNumHits":{0}}}]nearestNeighbor(content_embedding, query_embedding)) 
                or userQuery();
            """.format(k),
            "input.query(query_embedding)": query_text,
            "query": query_text,
            "ranking": "linear_fusion",
            "ranking.properties": {
                "bm25_weight": self.bm25_weight,
                "semantic_weight": self.semantic_weight
            },
            "hits": k
        }

        response = self.app.query(body=query_body)
        res_json = response.get_json()

        try:
            vespa_time = res_json["root"]["timing"]["total"]
        except KeyError:
            print("⚠️  [VespaLinearFusionRetriever.run] Warning: 'timing' not found.")
            vespa_time = -1.0

        hits = [(hit["fields"]["id"], hit.get("relevance", 1.0)) for hit in res_json.get("root", {}).get("children", [])]
        return hits, vespa_time





In [67]:
vespa_bm25 = VespaRetrieverlunas(ranking="bm25")    
vespa_semantic = VespaSemanticRetriever()
vespa_hybrid = VespaHybridRetriever()
vespa_linear_fusion = VespaLinearFusionRetriever(bm25_weight=0.5, semantic_weight=0.5)

In [55]:
# Test if any documents exist
response = vespa_bm25.app.query(body={
    "yql": "select * from sources * where true", 
    "hits": 1
})
total_docs = response.get_json().get('root', {}).get('fields', {}).get('totalCount', 0)
print(f"Total documents indexed: {total_docs}")

Total documents indexed: 278


In [56]:
# Try searching for common words
test_words = ["the", "and", "of", "to", "a"]
for word in test_words:
    response = vespa_bm25.app.query(body={
        "yql": f"select * from sources * where content contains '{word}'",
        "ranking": "bm25",
        "hits": 5
    })
    hits = len(response.get_json().get('root', {}).get('children', []))
    print(f"Word '{word}': {hits} hits")

Word 'the': 5 hits
Word 'and': 5 hits
Word 'of': 5 hits
Word 'to': 5 hits
Word 'a': 5 hits


In [49]:
a=queries_df.iloc[0]["text"]
a 

'how long till i can bath after a tattoo'

In [68]:
bm25_results, semantic_results, hybrid_results, fusion_results = {}, {}, {}, {}
bm25_times, semantic_times, hybrid_times, fusion_times = [], [], [], []

# Counters to verify that we get at least one hit
bm25_non_empty, semantic_non_empty = 0, 0
hybrid_non_empty, fusion_non_empty = 0, 0

for _, row in queries_df.iterrows():
    qid, qtext = row["query_id"], row["text"]

    # BM25
    bm25_hits, bm25_time = vespa_bm25.run(query_text=qtext, k=10)
    bm25_results[qid] = bm25_hits
    bm25_times.append(bm25_time)
    if len(bm25_hits) > 0:
        bm25_non_empty += 1

    # Semantic
    # sem_hits, sem_time = vespa_semantic.run(query_text=qtext, k=10)
    # semantic_results[qid] = sem_hits
    # semantic_times.append(sem_time)
    # if len(sem_hits) > 0:
    #     semantic_non_empty += 1

    # # Hybrid
    # hyb_hits, hyb_time = vespa_hybrid.run(query_text=qtext, k=10)
    # hybrid_results[qid] = hyb_hits
    # hybrid_times.append(hyb_time)
    # if len(hyb_hits) > 0:
    #     hybrid_non_empty += 1

    # # Fusion
    # fus_hits, fus_time = vespa_fusion.run(query_text=qtext, k=10)
    # fusion_results[qid] = fus_hits
    # fusion_times.append(fus_time)
    # if len(fus_hits) > 0:
    #     fusion_non_empty += 1

# Summary of non-empty results
total_queries = len(queries_df)

print("\n✅ Summary of non-empty results:")
print(f"BM25 - queries with hits: {bm25_non_empty}/{total_queries}")
# print(f"Semantic - queries with hits: {semantic_non_empty}/{total_queries}")
# print(f"Hybrid - queries with hits: {hybrid_non_empty}/{total_queries}")
# print(f"Fusion - queries with hits: {fusion_non_empty}/{total_queries}")

# Optional: flag if any method failed to return results
if bm25_non_empty < total_queries:
    print("⚠️ BM25 missed some queries.")
# if semantic_non_empty < total_queries:
#     print("⚠️ Semantic missed some queries.")
# if hybrid_non_empty < total_queries:
#     print("⚠️ Hybrid missed some queries.")
# if fusion_non_empty < total_queries:
#     print("⚠️ Fusion missed some queries.")



✅ Summary of non-empty results:
BM25 - queries with hits: 2752/2771
⚠️ BM25 missed some queries.


In [69]:

def eval_all(results, name):
    print(f"### {name}")
    print("MRR:", mrr_score(results, relevant_dict, k=10))
    print("MAP:", map_score(results, relevant_dict, k=10))
    print("Recall:", mr_score(results, relevant_dict, k=10))
    print("F1:", mf1_score(results, relevant_dict, k=10))
    print("nDCG:", mndcg_score(results, relevant_dict, k=10))
    print()

eval_all(bm25_results, "BM25")
eval_all(semantic_results, "Semantic")
eval_all(hybrid_results, "Hybrid")
eval_all(fusion_results, "Linear Fusion (alpha=0.6)")


### BM25
MRR: 0.01163839769036449
MAP: 0.001190905810176832
Recall: 0.011728617827499097
F1: 0.0021598154041315345
nDCG: 0.011482327450490052

### Semantic
MRR: 0
MAP: 0
Recall: 0
F1: 0
nDCG: 0

### Hybrid
MRR: 0
MAP: 0
Recall: 0
F1: 0
nDCG: 0

### Linear Fusion (alpha=0.6)
MRR: 0
MAP: 0
Recall: 0
F1: 0
nDCG: 0

